<a href="https://colab.research.google.com/github/gabrielcabanilla1/EXSU-500-Clinical-NLP-large-language-models-rct/blob/main/03aaaa_position_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
DATA_VERSION = 'PubMed_20k_RCT'
PROJECT_DIR = '/content/drive/MyDrive/exsu_project'
DATA_DIR = f'{PROJECT_DIR}/prepared_data'
MODELS_DIR = f'{PROJECT_DIR}/models'
PREDICTIONS_DIR = f'{PROJECT_DIR}/predictions'
RESULTS_DIR = f'{PROJECT_DIR}/results'
CLASSES = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']
SEED = 42
N_BOOTSTRAP = 2000
EXPECTED_SIZES = {'train': 180040, 'dev': 30212}

RUN_NAME = 'position_baseline'
BIN_CANDIDATES = [5, 10, 20]
SMOOTHING = 1
print('Run:', RUN_NAME, '| bins tried:', BIN_CANDIDATES)

Run: position_baseline | bins tried: [5, 10, 20]


In [8]:
import os
import json
import time
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
for folder in (MODELS_DIR, PREDICTIONS_DIR, RESULTS_DIR):
    os.makedirs(folder, exist_ok=True)

def load_split(split):
    return pd.read_csv(f'{DATA_DIR}/{DATA_VERSION}_{split}.csv',
                       dtype={'abstract_id': str}, keep_default_na=False)

train, dev = load_split('train'), load_split('dev')
for name, df in (('train', train), ('dev', dev)):
    assert len(df) == EXPECTED_SIZES[name], f'Unexpected {name} size: re-run 02_data_preparation'
    assert set(df.label) <= set(CLASSES), f'Unknown labels in {name}'
assert not set(train.abstract_id) & set(dev.abstract_id), 'Leakage: abstract in train and dev'
print(f'Train: {train.abstract_id.nunique():,} abstracts / {len(train):,} sentences | '
      f'Dev: {dev.abstract_id.nunique():,} abstracts / {len(dev):,} sentences')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: 15,000 abstracts / 180,040 sentences | Dev: 2,500 abstracts / 30,212 sentences


In [9]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, roc_auc_score,
                             average_precision_score, confusion_matrix)

def bootstrap_macro_f1(y_true, y_pred, abstract_ids, n_resamples=N_BOOTSTRAP, seed=SEED):
    index, unique_ids = pd.factorize(pd.Series(abstract_ids))
    k = len(CLASSES)
    counts = np.zeros((len(unique_ids), k, k), dtype=np.int64)
    np.add.at(counts, (index, y_true, y_pred), 1)
    rng = np.random.default_rng(seed)
    scores = np.empty(n_resamples)
    for b in range(n_resamples):
        cm = counts[rng.integers(0, len(unique_ids), len(unique_ids))].sum(axis=0)
        denominator = cm.sum(axis=0) + cm.sum(axis=1)
        scores[b] = np.divide(2 * np.diag(cm), denominator, out=np.zeros(k), where=denominator > 0).mean()
    return np.percentile(scores, [2.5, 97.5])

def evaluate_and_save(run_name, model_description, input_description, df, probs,
                      fit_seconds, hardware, settings):
    probs = np.asarray(probs, dtype=float)
    assert probs.shape == (len(df), len(CLASSES)), 'probs must be (n_sentences, 5) in CLASSES order'
    k = len(CLASSES)
    y_true = pd.Categorical(df.label, categories=CLASSES).codes
    y_pred = probs.argmax(axis=1)
    one_hot = np.eye(k)[y_true]

    cm = confusion_matrix(y_true, y_pred, labels=range(k))
    tp = np.diag(cm); fn = cm.sum(axis=1) - tp; fp = cm.sum(axis=0) - tp; tn = cm.sum() - tp - fn - fp
    per_class = pd.DataFrame({
        'precision': precision_score(y_true, y_pred, labels=range(k), average=None, zero_division=0),
        'sensitivity': tp / (tp + fn),
        'specificity': tn / (tn + fp),
        'f1': f1_score(y_true, y_pred, labels=range(k), average=None, zero_division=0),
        'auroc': [roc_auc_score(one_hot[:, i], probs[:, i]) for i in range(k)],
        'auprc': [average_precision_score(one_hot[:, i], probs[:, i]) for i in range(k)],
        'prevalence': one_hot.mean(axis=0),
    }, index=pd.Index(CLASSES, name='class'))
    per_class.loc['macro average'] = per_class.mean()

    lower, upper = bootstrap_macro_f1(y_true, y_pred, df.abstract_id)
    summary = {
        'run': run_name, 'model': model_description, 'input': input_description,
        'split': 'dev', 'abstracts': df.abstract_id.nunique(), 'sentences': len(df),
        'decision_rule': 'argmax',
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=range(k), average='macro', zero_division=0),
        'macro_f1_ci95_lower': lower, 'macro_f1_ci95_upper': upper,
        'weighted_f1': f1_score(y_true, y_pred, labels=range(k), average='weighted', zero_division=0),
        'macro_auroc': per_class.loc['macro average', 'auroc'],
        'macro_auprc': per_class.loc['macro average', 'auprc'],
        'macro_sensitivity': per_class.loc['macro average', 'sensitivity'],
        'macro_specificity': per_class.loc['macro average', 'specificity'],
        'fit_seconds': round(float(fit_seconds), 1), 'hardware': hardware,
        'seed': SEED, 'n_bootstrap': N_BOOTSTRAP, 'settings': json.dumps(settings),
    }

    predictions = df[['abstract_id', 'sentence_idx', 'label']].copy()
    predictions['prediction'] = np.array(CLASSES)[y_pred]
    for i, c in enumerate(CLASSES):
        predictions[f'prob_{c}'] = probs[:, i]
    predictions.to_csv(f'{PREDICTIONS_DIR}/dev_{run_name}.csv', index=False)
    pd.DataFrame([summary]).to_csv(f'{RESULTS_DIR}/dev_{run_name}_summary.csv', index=False)
    per_class.to_csv(f'{RESULTS_DIR}/dev_{run_name}_per_class.csv')
    confusion = pd.DataFrame(cm, index=pd.Index(CLASSES, name='actual'),
                             columns=pd.Index(CLASSES, name='predicted'))
    confusion.to_csv(f'{RESULTS_DIR}/dev_{run_name}_confusion_matrix.csv')

    print(f"{model_description} | {input_description}")
    print(f"Dev macro-F1 {summary['macro_f1']:.4f} (95% CI {lower:.4f}-{upper:.4f}) | "
          f"accuracy {summary['accuracy']:.4f} | weighted F1 {summary['weighted_f1']:.4f} | "
          f"macro AUROC {summary['macro_auroc']:.4f} | macro AUPRC {summary['macro_auprc']:.4f}")
    print(f'Saved predictions and 3 result tables for run "{run_name}"')
    print('\nPer-class metrics (one class vs. the other four; decision rule: argmax):')
    display(per_class.round(4))
    print('\nConfusion matrix (rows = actual class, columns = predicted class):')
    display(confusion)
    never_predicted = [c for i, c in enumerate(CLASSES) if (y_pred == i).sum() == 0]
    print('Classes never predicted:', never_predicted or 'none')
    return summary, per_class

In [10]:
def position_bins(df, n_bins):
    return np.minimum((df.position * n_bins).astype(int), n_bins - 1)

def fit_position_table(n_bins):
    counts = (pd.crosstab(position_bins(train, n_bins), train.label)
                .reindex(index=range(n_bins), columns=CLASSES, fill_value=0))
    smoothed = counts + SMOOTHING
    return counts, smoothed.div(smoothed.sum(axis=1), axis=0)

selection = []
for n_bins in BIN_CANDIDATES:
    _, table = fit_position_table(n_bins)
    probs = table.loc[position_bins(dev, n_bins)].to_numpy()
    selection.append({'n_bins': n_bins,
                      'dev_macro_f1': f1_score(dev.label, np.array(CLASSES)[probs.argmax(axis=1)],
                                               labels=CLASSES, average='macro', zero_division=0)})
selection = pd.DataFrame(selection)
display(selection.round(4))

N_BINS = int(selection.loc[selection.dev_macro_f1.idxmax(), 'n_bins'])
start = time.time()
counts, probability_table = fit_position_table(N_BINS)
fit_seconds = time.time() - start
dev_probs = probability_table.loc[position_bins(dev, N_BINS)].to_numpy()

rule = probability_table.idxmax(axis=1)
probability_table.assign(predicted_label=rule).to_csv(f'{MODELS_DIR}/{RUN_NAME}_table.csv')
print(f'Chosen on the validation set: {N_BINS} bins | fitted in {fit_seconds:.3f} s')
print('Training sentences per bin and label, with the label the rule predicts:')
display(counts.assign(predicted_label=rule))
print('Classes the rule never predicts:', [c for c in CLASSES if c not in set(rule)] or 'none')

,n_bins,dev_macro_f1
0,5,0.5079
1,10,0.5477
2,20,0.6088


Chosen on the validation set: 20 bins | fitted in 0.020 s
Training sentences per bin and label, with the label the rule predicts:


label,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS,predicted_label
position,,,,,,
0,7453,7537,97,0,14,OBJECTIVE
1,3830,1868,2164,27,3,BACKGROUND
2,2976,2002,4151,92,5,METHODS
3,1523,884,4187,133,0,METHODS
4,1175,769,5824,235,0,METHODS
5,688,424,7371,444,0,METHODS
6,377,193,7700,910,1,METHODS
7,143,65,5510,1162,5,METHODS
8,139,47,6935,2629,8,METHODS


Classes the rule never predicts: none


In [11]:
summary, per_class = evaluate_and_save(
    RUN_NAME, f'Position baseline ({N_BINS} bins)', 'position only',
    dev, dev_probs, fit_seconds, f'CPU ({os.cpu_count()} cores)',
    {'n_bins': N_BINS, 'bins_tried': BIN_CANDIDATES, 'smoothing': SMOOTHING})

Position baseline (20 bins) | position only
Dev macro-F1 0.6088 (95% CI 0.6029-0.6147) | accuracy 0.6923 | weighted F1 0.6756 | macro AUROC 0.9158 | macro AUPRC 0.6252
Saved predictions and 3 result tables for run "position_baseline"

Per-class metrics (one class vs. the other four; decision rule: argmax):


,precision,sensitivity,specificity,f1,auroc,auprc,prevalence
class,,,,,,,
BACKGROUND,0.4521,0.1780,0.9722,0.2555,0.8831,0.4110,0.1142
OBJECTIVE,0.5131,0.5438,0.9560,0.5280,0.9328,0.4413,0.0786
METHODS,0.7097,0.7559,0.8478,0.7321,0.8913,0.7504,0.3298
RESULTS,0.7389,0.7747,0.8678,0.7564,0.9090,0.7781,0.3257
CONCLUSIONS,0.7132,0.8413,0.9395,0.7720,0.9627,0.7452,0.1517
macro average,0.6254,0.6188,0.9167,0.6088,0.9158,0.6252,0.2000



Confusion matrix (rows = actual class, columns = predicted class):


predicted,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS
actual,,,,,
BACKGROUND,614,1205,1093,25,512
OBJECTIVE,354,1292,728,2,0
METHODS,383,18,7532,1949,82
RESULTS,7,0,1254,7624,956
CONCLUSIONS,0,3,6,718,3855


Classes never predicted: none


In [12]:
majority = train.label.value_counts().idxmax()
majority_pred = np.full(len(dev), majority)
print(f'Always "{majority}": accuracy {accuracy_score(dev.label, majority_pred):.4f}, '
      f'macro-F1 {f1_score(dev.label, majority_pred, labels=CLASSES, average="macro", zero_division=0):.4f}')

Always "METHODS": accuracy 0.3298, macro-F1 0.0992
